In [1]:
# Assignment 4D: Recommendation System using CBOW

# 1. Import libraries
import pandas as pd
import re
import numpy as np

from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# 2. Load dataset
df = pd.read_csv("twitter_dataset.csv")

print("Dataset Shape:", df.shape)
print(df.head())

#---------------------------------------------------------
# 3. Preprocessing
stop_words = set(ENGLISH_STOP_WORDS)

def preprocess(text):
    text = str(text).lower()

    # Remove URLs and usernames
    text = re.sub(r"http\S+|www\S+|@\w+", " ", text)

    # Remove special characters and numbers
    text = re.sub(r"[^a-z\s]", " ", text)

    # Tokenization
    words = text.split()

    # Remove stopwords
    clean_words = []

    for word in words:
        if word not in stop_words:
            clean_words.append(word)

    return clean_words


# 4. Apply preprocessing to tweets
tweets = []
original_tweets = []

for text in df["Text"].dropna():
    cleaned = preprocess(text)

    if len(cleaned) > 0:
        tweets.append(cleaned)
        original_tweets.append(text)

print("\nTotal Tweets:", len(tweets))
print("Sample Processed Tweet:", tweets[0])
#------------------------------------------------

# 5. Train CBOW Word2Vec model
model = Word2Vec(
    sentences=tweets,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,                 # CBOW
    epochs=20
)

print("\nCBOW Model Trained Successfully!")

#--------------------------------------------------
# 6. Convert tweet into average word embedding
def get_vector(words):
    vectors = []

    for word in words:
        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)


# 7. Create vectors for all tweets
tweet_vectors = []

for tweet in tweets:
    tweet_vectors.append(get_vector(tweet))

tweet_vectors = np.array(tweet_vectors)
#--------------------------------------------------

# 8. Take user input
user_input = input("\nEnter your search text: ")

user_tokens = preprocess(user_input)

user_vector = get_vector(user_tokens)


# 9. Check known words
known_words = False

for word in user_tokens:
    if word in model.wv:
        known_words = True
        break

if known_words == False:
    print("No known words found. Please enter different text.")

else:

    # 10. Calculate cosine similarity
    similarities = cosine_similarity(
        user_vector.reshape(1, -1),
        tweet_vectors
    )[0]


    # 11. Get top 5 recommendations
    top_indices = np.argsort(similarities)[::-1][:5]

    print("\nRecommended Tweets:\n")

    for i in range(5):
        idx = top_indices[i]
    
        print("Recommendation", i + 1)
        print("Tweet:", original_tweets[idx])
        print("Similarity:", round(similarities[idx], 4))
        print("----------------------")

Dataset Shape: (10000, 6)
   Tweet_ID        Username  \
0         1         julie81   
1         2   richardhester   
2         3  williamsjoseph   
3         4     danielsmary   
4         5      carlwarren   

                                                Text  Retweets  Likes  \
0  Party least receive say or single. Prevent pre...         2     25   
1  Hotel still Congress may member staff. Media d...        35     29   
2  Nice be her debate industry that year. Film wh...        51     25   
3  Laugh explain situation career occur serious. ...        37     18   
4  Involve sense former often approach government...        27     80   

             Timestamp  
0  2023-01-30 11:00:51  
1  2023-01-02 22:45:58  
2  2023-01-18 11:25:19  
3  2023-04-10 22:06:29  
4  2023-01-24 07:12:21  

Total Tweets: 10000
Sample Processed Tweet: ['party', 'receive', 'say', 'single', 'prevent', 'prevent', 'husband', 'affect', 'cup', 'style', 'evening', 'protect', 'effect', 'stage', 'perform', 'pos


Enter your search text:  party



Recommended Tweets:

Recommendation 1
Tweet: In ask mean foreign pass. Wrong sing thousand one. Religious pass party always early.
Learn offer off parent everyone. Art street law. Respond discussion answer else cultural success.
Similarity: 0.5456
----------------------
Recommendation 2
Tweet: Executive executive everyone tonight. Material him crime life property five great.
Too sing but once. Fill camera close smile future party or. Mission third process federal today.
Similarity: 0.521
----------------------
Recommendation 3
Tweet: President meeting available give true thousand hear. Society prevent rock smile.
Free wait very fine result field so.
Sea discussion region hit party pattern just. Form detail its have professional anyone least.
Similarity: 0.5137
----------------------
Recommendation 4
Tweet: Want leave still. Husband go result party soon whether season so. Former value Mrs.
Network as within situation worker. Respond kid south response.
Similarity: 0.5109
--------------